In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import itertools

In [ ]:
import boto3
import s3fs

In [ ]:
DATASETS = ["ptbxl_superclasses", "ptbxl_subclasses", "ptbxl_form", "ptbxl_rhythm", "cpsc2018", "cs", "csn"]  # ["ptbxl", "cpsc2018", "cs", "csn"]
USEFRACS = ["0.01", "0.1", "1.0"]     # [0.01, 0.1, 1.0]
NLEADS = 12
bucket_out = "walkky-ml"

s3_client = boto3.client("s3")
s3_fs = s3fs.S3FileSystem()

In [ ]:
# Test results

for dataset, usefrac in itertools.product(DATASETS, USEFRACS):
    prefix_out_tst = f"aruna-files/vqvae/bert_finetuning/{dataset}/finetune_test_metrics_{dataset}_{usefrac}.npz"
    #prefix_out_tst = f"aruna-files/vqvae_final_12lead_nocnn/vqvae/bert_finetuning_seed1/bert_finetuning/{dataset}/finetune_test_metrics_{dataset}_{usefrac}.npz"
    #prefix_out_tst = f"aruna-files/vqvae_final_12lead_vqenc/vqvae/bert_finetuning_seed1/bert_finetuning/{dataset}/finetune_test_metrics_{dataset}_{usefrac}.npz"
    print("====================================")
    print(f"Dataset {dataset}, use frac {usefrac}")
    print("====================================")
    try:
        with s3_fs.open(f"s3://{bucket_out}/{prefix_out_tst}", "rb") as s3_file:
            # Load the compressed data archive
            with np.load(s3_file) as data:
                print(f"Test AUROC = {data['test_auroc']}")
                print(f"Test loss = {data['test_loss']}")
                print(f"Test accuracy = {data['test_accuracy']}")
                print(f"Test F1 score = {data['test_f1']}")
                print(f"Test record-level AUROC = {data['test_record_auroc']}")
                print(f"Test record-level AUROC mean = {np.mean(data['test_record_auroc'])*100:.2f}, std = {np.std(data['test_record_auroc'])*100:.2f}")
                print(f"Seed = {data['seeds']}")
    except Exception as e:
        print(e)

In [ ]:
# Training and validation results and plots
def train_val_plots(dataset):
    fig, ax = plt.subplots(3, 4, figsize=(12,8))
    plt.subplots_adjust(hspace=0.3)
    fig.suptitle(f"{dataset}")
    for (i, usefrac) in enumerate(USEFRACS):
        prefix_out_trn = f"aruna-files/vqvae/bert_finetuning/{dataset}/finetune_train_val_metrics_{dataset}_{usefrac}_seed42.npz"
        with s3_fs.open(f"s3://{bucket_out}/{prefix_out_trn}", "rb") as s3_file:
            # Load the compressed data archive
            with np.load(s3_file) as data:
                
                epochs = data["epoch"]
                train_loss = data["train_loss"]
                train_accuracy = data["train_accuracy"]
                train_f1 = data["train_f1"]
                train_auroc = data["train_auroc"]
                val_loss = data["val_loss"]
                val_accuracy = data["val_accuracy"]
                val_f1 = data["val_f1"]
                val_auroc = data["val_auroc"]
    
                ax[i, 0].plot(epochs, train_loss, label='train loss')
                ax[i, 0].plot(epochs, val_loss, label='val loss')
                ax[i, 0].set_title(f"Use frac = {usefrac}")
                ax[i, 0].legend()
    
                ax[i, 1].plot(epochs, train_accuracy, label='train accuracy')
                ax[i, 1].plot(epochs, val_accuracy, label='val accuracy')
                ax[i, 1].legend()
    
                ax[i, 2].plot(epochs, train_f1, label='train F1')
                ax[i, 2].plot(epochs, val_f1, label='val F1')
                ax[i, 2].legend()
    
                ax[i, 3].plot(epochs, train_auroc, label='train AUROC')
                ax[i, 3].plot(epochs, val_auroc, label='val AUROC')
                ax[i, 3].legend()

                print(f" Dataset {dataset}, use_frac {usefrac}: epochs: {epochs}, train_auroc: {train_auroc}, "
                      f" val_auroc: {val_auroc}, train_loss: {train_loss}, val_loss: {val_loss} ")
                

In [ ]:
for dataset in DATASETS:
    train_val_plots(dataset)

In [ ]:
# Test results summary, mean and std of sequence-level AUROC

for dataset, usefrac in itertools.product(DATASETS, USEFRACS):
    prefix_out_summary_tst = f"aruna-files/vqvae/bert_finetuning/{dataset}/finetune_test_summary_{dataset}_{usefrac}.npz"
    print("====================================")
    print(f"Dataset {dataset}, use frac {usefrac}")
    print("====================================")
    try:
        with s3_fs.open(f"s3://{bucket_out}/{prefix_out_summary_tst}", "rb") as s3_file:
            # Load the compressed data archive
            with np.load(s3_file) as data:
                print(f"Seeds: {data['seeds']}")
                print(f"Sequence-level Test AUROC: {data['test_auroc']}")
                print(f"Sequence-level Mean test AUROC: {data['mean_test_auroc'][0]*100: .2f}")
                print(f"Sequence-level Std test AUROC: {data['std_test_auroc'][0]*100: .2f}")
    except Exception as e:
        print(e)